In [ ]:
"""
cross_temporal_decoder.py
=========================
Cross-temporal decoding: train a decoder on one time bin, test on all others.
Produces an (n_bins × n_bins) generalisation matrix per contrast × alignment.

Key difference from standard decoding:
  - Standard : train on bin b, test on bin b  → diagonal only
  - Cross-temporal: train on bin b_train, test on bin b_test → full matrix

Normalisation uses train-bin statistics (mean/max computed on train set only),
applied to both train and test data — prevents data leakage across bins.

Usage
-----
from cross_temporal_decoder import cross_temporal_decode

mat = cross_temporal_decode(
    spk_count,        # shape (n_bins, n_trials, n_neurons)
    labels,           # shape (n_trials,)  — 0/1 class labels
    ratio_train_val=0.8, ncv=50, n_jobs=-1
)
# mat: (n_bins_train, n_bins_test) balanced accuracy matrix
"""

import numpy as np
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix
from joblib import Parallel, delayed


# ---------------------------------------------------------------------------
# Shared utilities
# ---------------------------------------------------------------------------

def _bac(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    if (tn + fp) == 0:
        return tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    if (tp + fn) == 0:
        return tn / (tn + fp) if (tn + fp) > 0 else float('nan')
    return ((tp / (tp + fn)) + (tn / (tn + fp))) / 2.0


def _normalise_with_train_params(X_train, X_test):
    """
    Normalise X_train and X_test using ONLY X_train statistics.
    Prevents leakage: test-bin data is scaled with train-bin parameters.
    """
    mean = np.mean(X_train, axis=0)
    mx   = np.max(X_train,  axis=0)
    denom = np.where(mx == 0, 1.0, mx)

    X_train_n = (X_train - mean) / denom
    X_test_n  = (X_test  - mean) / denom

    # Remove columns that are all-NaN in the train set
    valid = ~np.all(np.isnan(X_train_n), axis=0)
    return X_train_n[:, valid], X_test_n[:, valid]


def _build_classifier(method, Cvec=None):
    """Return a fitted-classifier factory for the given method."""
    if method == 'svm':
        # C is fixed to 1.0 for cross-temporal (no inner CV to keep it fast).
        # To use optimal C, pass the C found by standard decoding per bin.
        return SVC(kernel='linear', C=1.0)
    elif method == 'lda':
        return LinearDiscriminantAnalysis(solver='svd')
    elif method == 'nb':
        return GaussianNB()
    else:
        raise ValueError(f"Unknown method: {method}")


# ---------------------------------------------------------------------------
# One Monte-Carlo repeat for the full cross-temporal matrix
# ---------------------------------------------------------------------------

def _one_cv(spk_count, labels, n_bins, n_trials,
            ntrain, nval, method):
    """
    One MC repeat.
    Returns a (n_bins, n_bins) matrix: mat[b_train, b_test] = BAC.
    """
    # Random train/val split of TRIALS (same split used for all bin pairs)
    rp = np.random.permutation(n_trials)
    train_idx = rp[:ntrain]
    val_idx   = rp[ntrain:]

    # Labels for train and validation
    label_train = labels[train_idx]
    label_val   = labels[val_idx]

    # Guard: need at least 1 sample of each class in both sets
    if (len(np.unique(label_train)) < 2 or
            len(np.unique(label_val)) < 2):
        return np.full((n_bins, n_bins), float('nan'))

    mat = np.full((n_bins, n_bins), float('nan'))

    for b_train in range(n_bins):
        X_all_train_bin = spk_count[b_train]   # (n_trials, n_neu)
        X_tr_raw = X_all_train_bin[train_idx]  # training trials, train bin
        X_va_raw = X_all_train_bin[val_idx]    # val trials,   train bin (same bin for norm)

        # Normalise using train-bin, train-trials statistics
        X_tr, _ = _normalise_with_train_params(X_tr_raw, X_va_raw)

        # Fit classifier on train-bin data
        try:
            clf = _build_classifier(method)
            clf.fit(X_tr, label_train)
        except Exception:
            continue

        for b_test in range(n_bins):
            # Get test-bin data for validation trials
            X_test_raw = spk_count[b_test][val_idx]

            # Apply train-bin normalisation to test-bin data
            _, X_te = _normalise_with_train_params(X_tr_raw, X_test_raw)

            try:
                pred = clf.predict(X_te)
                mat[b_train, b_test] = _bac(label_val, pred)
            except Exception:
                pass

    return mat


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def cross_temporal_decode(spk_count, labels,
                          ratio_train_val=0.8, ncv=50,
                          method='svm', n_jobs=-1,
                          min_trials_per_class=5):
    """
    Cross-temporal decoding generalisation matrix.

    Parameters
    ----------
    spk_count : np.ndarray, shape (n_bins, n_trials, n_neurons)
        Spike counts per bin, trial, neuron.
    labels : np.ndarray, shape (n_trials,)
        Binary class labels (0 and 1).
    ratio_train_val : float
        Fraction of trials for training.
    ncv : int
        Number of Monte-Carlo CV repeats.
    method : str
        'svm', 'lda', or 'nb'.
    n_jobs : int
        Parallel jobs (-1 = all cores).
    min_trials_per_class : int
        Minimum trials per class; returns NaN matrix if not met.

    Returns
    -------
    mat : np.ndarray, shape (n_bins, n_bins)
        Mean BAC generalisation matrix (nanmean across CV repeats).
        mat[b_train, b_test] = BAC when trained on b_train, tested on b_test.
        Diagonal = standard decoding accuracy.
    """
    labels = np.asarray(labels, dtype=int)
    n_bins, n_trials, _ = spk_count.shape

    # Guard
    classes, counts = np.unique(labels, return_counts=True)
    if len(classes) < 2:
        return np.full((n_bins, n_bins), float('nan'))
    if counts.min() < min_trials_per_class:
        return np.full((n_bins, n_bins), float('nan'))

    ntrain = int(np.floor(n_trials * ratio_train_val))
    nval   = n_trials - ntrain

    if ntrain < 2 or nval < 1:
        return np.full((n_bins, n_bins), float('nan'))

    # Parallelise over MC repeats
    mats = Parallel(n_jobs=n_jobs, prefer='threads')(
        delayed(_one_cv)(
            spk_count, labels, n_bins, n_trials,
            ntrain, nval, method
        )
        for _ in range(ncv)
    )

    return np.nanmean(np.stack(mats, axis=0), axis=0)  # (n_bins, n_bins)

In [ ]:
"""
run_cross_temporal.py
=====================

Cross-temporal decoding directly from raw .mat files.

This version DOES NOT rely on raw spike counts stored in the npz.
Instead it reconstructs the spike-count tensors from the original
.mat files exactly like add_kalman.py.

Outputs
-------
cross_temporal/
    comparison/
        grand_{method}_{contrast}.png
        BGvsCB_{method}_{contrast}_{alignment}.png
        diagonal_BG_CB_{contrast}.png

    stats/
        cross_temporal_stats.csv
"""

# ===========================================================================
# Imports
# ===========================================================================

import os
import time
import numpy as np
import scipy.io as sio
import h5py
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.lines as mlines

from scipy import stats
from joblib import Parallel, delayed

from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import balanced_accuracy_score


# ===========================================================================
# Configuration
# ===========================================================================

rootdir  = r'path/to/ephys_data'  # ephys_and_behavior/mat_files directory
group    = '20230801_ChocolateGroup'
base_mat = os.path.join(rootdir, 'ephys_and_behavior', 'mat_files', group)

BG_NPZ = r'data/SVM_sess_BG.npz'
CB_NPZ = r'data/SVM_sess_CB.npz'

SAVE_DIR = r'output/cross_temporal'

os.makedirs(SAVE_DIR, exist_ok=True)
for sub in ['comparison', 'stats']:
    os.makedirs(os.path.join(SAVE_DIR, sub), exist_ok=True)

animals  = ['CoteDor', 'Lindt', 'Toblerone', 'Milka', 'FerreroRocher']

all_sess = [
    ['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7'],
    ['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7'],
    ['R1', 'R2', 'R3', 'R4', 'R5', 'R6'],
    ['R1', 'R4', 'R5', 'R6', 'R7'],
    ['R1', 'R4', 'R5', 'R6'],
]

# ===========================================================================
# Decoder parameters
# ===========================================================================

METHODS         = ['svm', 'lda', 'nb']

METHOD_LABELS = {
    'svm': 'SVM',
    'lda': 'LDA',
    'nb':  'Naive Bayes'
}

CONTRASTS = ['pp', 'D', 'C', 'nD']

CONTRAST_LABELS = {
    'pp': 'Push/Pull',
    'D':  'Dominant',
    'C':  'Center',
    'nD': 'Non-dominant'
}

ALIGNMENTS = [0, 1]

ALIGN_LABELS = {
    0: 'Init',
    1: 'Reach'
}

CB_CELL_TYPE = 'all'

RATIO_TRAIN_VAL = 0.8
NCV             = 50
N_JOBS          = -1
MIN_TRIALS      = 10

# ===========================================================================
# Sliding window
# ===========================================================================

win_int   = [-2.5, 2.5]
bin_width = 0.1
bin_step  = 0.025

tm_before, tm_after = win_int

win_dur = tm_after - tm_before

n_bins = int(
    np.floor((win_dur / bin_step) - (bin_width / bin_step)) + 1
)

bin_edges = np.array([
    [
        tm_before + i * bin_step,
        tm_before + bin_width + i * bin_step
    ]
    for i in range(n_bins)
])

bw = bin_edges[0, 1] - bin_edges[0, 0]
bin_mean = bin_edges[:, 0] + bw / 2


# ===========================================================================
# Shared loaders
# ===========================================================================

def _arr(val):
    return np.atleast_1d(np.squeeze(np.asarray(val))).ravel()


def _load_behavior(filepath):
    data = sio.loadmat(filepath, simplify_cells=True)
    return data['bhv']


def _n_neu(filepath):
    with h5py.File(filepath, 'r') as f:
        return f['eg_neurons']['st_init'].shape[1]


def _load_spike_times(filepath, n_neu):

    st_init  = []
    st_reach = []

    with h5py.File(filepath, 'r') as f:

        eg = f['eg_neurons']

        for n in range(n_neu):

            ds_i = f[eg['st_init'][0, n]]
            ds_r = f[eg['st_reach'][0, n]]

            st_init.append([
                f[ds_i[tt, 0]][()].ravel().astype(float)
                for tt in range(ds_i.shape[0])
            ])

            st_reach.append([
                f[ds_r[tt, 0]][()].ravel().astype(float)
                for tt in range(ds_r.shape[0])
            ])

    return st_init, st_reach


def _read_hdf5_str(ds, f):

    try:
        data = ds[()]

        if data.dtype == object:
            return ''.join(
                chr(int(f[r][()])) for r in data.ravel()
            ).strip()

        return ''.join(chr(c) for c in data.ravel()).strip()

    except Exception:
        return 'NaN'


def _load_cell_types_CB(filepath):

    cell_types = []

    with h5py.File(filepath, 'r') as f:

        eg    = f['eg_neurons']
        n_neu = eg['st_init'].shape[1]

        ct_ds = eg['cell_type']

        for n in range(n_neu):

            ct = _read_hdf5_str(f[ct_ds[0, n]], f)

            if not ct or ct.lower() in ('nan', ''):
                ct = 'NaN'

            cell_types.append(ct)

    return cell_types, n_neu


def _build_spk_count(st_list, n_trials, n_neu):

    sc = np.zeros((n_bins, n_trials, n_neu), dtype=np.float32)

    for n_idx in range(n_neu):

        for tt in range(n_trials):

            spk = np.atleast_1d(
                st_list[n_idx][tt]
            ).ravel()

            for i, (bs, be) in enumerate(bin_edges):

                sc[i, tt, n_idx] = np.sum(
                    (spk > bs) & (spk < be)
                )

    return sc


# ===========================================================================
# Cross-temporal decoder
# ===========================================================================

def _make_decoder(method):

    if method == 'svm':
        return SVC(kernel='linear')

    elif method == 'lda':
        return LinearDiscriminantAnalysis()

    elif method == 'nb':
        return GaussianNB()

    else:
        raise ValueError(method)


def cross_temporal_decode(
    spk,
    labels,
    ratio_train_val=0.8,
    ncv=50,
    method='svm'
):
    """
    spk shape:
        (n_bins, n_trials, n_neurons)
    """

    spk = np.asarray(spk, dtype=float)

    n_bins, n_trials, n_neu = spk.shape

    idx0 = np.where(labels == 0)[0]
    idx1 = np.where(labels == 1)[0]

    n0 = len(idx0)
    n1 = len(idx1)

    if n0 < MIN_TRIALS or n1 < MIN_TRIALS:
        return None

    ntr0 = int(np.floor(n0 * ratio_train_val))
    ntr1 = int(np.floor(n1 * ratio_train_val))

    mats = []

    for cv in range(ncv):

        rp0 = np.random.permutation(idx0)
        rp1 = np.random.permutation(idx1)

        tr0 = rp0[:ntr0]
        va0 = rp0[ntr0:]

        tr1 = rp1[:ntr1]
        va1 = rp1[ntr1:]

        train_idx = np.concatenate([tr0, tr1])
        test_idx  = np.concatenate([va0, va1])

        y_train = labels[train_idx]
        y_test  = labels[test_idx]

        mat = np.full((n_bins, n_bins), np.nan)

        for btrain in range(n_bins):

            X_train = spk[btrain, train_idx, :]

            mean = np.mean(X_train, axis=0)
            std  = np.std(X_train, axis=0)

            std[std == 0] = 1.0

            X_train = (X_train - mean) / std

            clf = _make_decoder(method)

            try:
                clf.fit(X_train, y_train)
            except Exception:
                continue

            for btest in range(n_bins):

                X_test = spk[btest, test_idx, :]
                X_test = (X_test - mean) / std

                try:
                    pred = clf.predict(X_test)

                    bac = balanced_accuracy_score(
                        y_test,
                        pred
                    )

                    mat[btrain, btest] = bac

                except Exception:
                    continue

        mats.append(mat)

    return np.nanmean(np.stack(mats, axis=0), axis=0)


# ===========================================================================
# Session loader
# ===========================================================================

def load_session_pairs(sess, region='BG', cb_cell_type='all'):

    mouse  = sess['mouse']
    s_name = sess['sess']

    sess_base = os.path.join(base_mat, mouse, s_name)

    ephys_path = os.path.join(
        sess_base,
        f'eg_neurons_{region}.mat'
    )

    behav_path = os.path.join(
        sess_base,
        'behavior_fundamentals.mat'
    )

    if not os.path.isfile(ephys_path):
        return None

    if not os.path.isfile(behav_path):
        return None

    # -----------------------------------------------------------------------
    # Behavior
    # -----------------------------------------------------------------------

    bhv = _load_behavior(behav_path)

    inv_pp = _arr(
        bhv['init_invPush_invPull_idx']
    ).astype(int)

    i_pp = np.where(
        inv_pp == 1,
        0,
        np.where(inv_pp == 2, 1, 3)
    )

    i_DCnD = _arr(
        bhv['DCnD_init']
    ).astype(int)

    r_pp = _arr(
        bhv['pp_inVec_idx']
    ).astype(int)

    r_DCnD = _arr(
        bhv['DCnD_inVec_idx']
    ).astype(int)

    r_hit = np.nan_to_num(
        np.atleast_1d(
            bhv['hit_inVec']
        ).ravel().astype(float),
        nan=0.0
    ).astype(int)

    # -----------------------------------------------------------------------
    # Spikes
    # -----------------------------------------------------------------------

    n_neu = _n_neu(ephys_path)

    st_init, st_reach = _load_spike_times(
        ephys_path,
        n_neu
    )

    sc_init = _build_spk_count(
        st_init,
        len(i_pp),
        n_neu
    )

    sc_reach = _build_spk_count(
        st_reach,
        len(r_hit),
        n_neu
    )

    # -----------------------------------------------------------------------
    # Cell-type filtering
    # -----------------------------------------------------------------------

    if region == 'CB':

        cell_types, _ = _load_cell_types_CB(
            ephys_path
        )

        ct_arr = np.array(cell_types)

        if cb_cell_type != 'all':

            mask = ct_arr == cb_cell_type

            if mask.sum() == 0:
                return None

            sc_init  = sc_init[:, :, mask]
            sc_reach = sc_reach[:, :, mask]

    # -----------------------------------------------------------------------
    # Build condition pairs
    # -----------------------------------------------------------------------

    hit = r_hit == 1

    pairs_init = {
        'pp': (
            sc_init[:, i_pp == 0, :],
            sc_init[:, i_pp == 1, :]
        ),

        'D': (
            sc_init[:, i_DCnD == 1, :],
            sc_init[:, (i_DCnD == 2) | (i_DCnD == 3), :]
        ),

        'C': (
            sc_init[:, i_DCnD == 2, :],
            sc_init[:, (i_DCnD == 1) | (i_DCnD == 3), :]
        ),

        'nD': (
            sc_init[:, i_DCnD == 3, :],
            sc_init[:, (i_DCnD == 1) | (i_DCnD == 2), :]
        ),
    }

    pairs_reach = {
        'pp': (
            sc_reach[:, (r_pp == 0) & hit, :],
            sc_reach[:, (r_pp == 1) & hit, :]
        ),

        'D': (
            sc_reach[:, (r_DCnD == 1) & hit, :],
            sc_reach[:, ((r_DCnD == 2) | (r_DCnD == 3)) & hit, :]
        ),

        'C': (
            sc_reach[:, (r_DCnD == 2) & hit, :],
            sc_reach[:, ((r_DCnD == 1) | (r_DCnD == 3)) & hit, :]
        ),

        'nD': (
            sc_reach[:, (r_DCnD == 3) & hit, :],
            sc_reach[:, ((r_DCnD == 1) | (r_DCnD == 2)) & hit, :]
        ),
    }

    return pairs_init, pairs_reach


# ===========================================================================
# Main processing
# ===========================================================================

def run_region(sessions, region_name, cb_cell_type='all'):

    results = {}
    animals_in_data = []

    for sess in sessions:

        animal = sess['mouse']

        if animal not in animals_in_data:
            animals_in_data.append(animal)

        if animal not in results:

            results[animal] = {
                m: {
                    c: {
                        a: []
                        for a in ALIGNMENTS
                    }
                    for c in CONTRASTS
                }
                for m in METHODS
            }

        print(f'\n[{region_name}] {animal} {sess["sess"]}')

        t0 = time.time()

        loaded = load_session_pairs(
            sess,
            region=region_name,
            cb_cell_type=cb_cell_type
        )

        if loaded is None:
            print('  failed loading')
            continue

        pairs_init, pairs_reach = loaded

        for contrast in CONTRASTS:

            for align in ALIGNMENTS:

                if align == 0:
                    s1, s2 = pairs_init[contrast]
                else:
                    s1, s2 = pairs_reach[contrast]

                n1 = s1.shape[1]
                n2 = s2.shape[1]

                print(
                    f'  {contrast} {ALIGN_LABELS[align]} '
                    f'n1={n1} n2={n2}'
                )

                if n1 < MIN_TRIALS or n2 < MIN_TRIALS:

                    for method in METHODS:
                        results[animal][method][contrast][align].append(None)

                    continue

                spk = np.concatenate([s1, s2], axis=1)

                labels = np.array(
                    [0] * n1 + [1] * n2
                )

                for method in METHODS:

                    print(f'    {method}...', end='')

                    mat = cross_temporal_decode(
                        spk,
                        labels,
                        ratio_train_val=RATIO_TRAIN_VAL,
                        ncv=NCV,
                        method=method
                    )

                    results[animal][method][contrast][align].append(mat)

                    print(' done')

        print(f'  session time: {time.time()-t0:.1f}s')

    return results, animals_in_data


# ===========================================================================
# Load sessions
# ===========================================================================

print('Loading npz metadata...')

bg_sessions = np.load(
    BG_NPZ,
    allow_pickle=True
)['SVM_sess'].tolist()

cb_sessions = np.load(
    CB_NPZ,
    allow_pickle=True
)['SVM_sess'].tolist()

print(f'BG sessions: {len(bg_sessions)}')
print(f'CB sessions: {len(cb_sessions)}')


# ===========================================================================
# Run decoding
# ===========================================================================

print('\n=== BG ===')

bg_results, bg_animals = run_region(
    bg_sessions,
    'BG'
)

print('\n=== CB ===')

cb_results, cb_animals = run_region(
    cb_sessions,
    'CB',
    cb_cell_type=CB_CELL_TYPE
)


# ===========================================================================
# Grand mean helper
# ===========================================================================

def grand_mean_matrix(results, animals, method, contrast, align):

    animal_means = []

    for animal in animals:

        sess_mats = results[animal][method][contrast][align]

        valid = [
            m for m in sess_mats
            if m is not None
        ]

        if len(valid) == 0:
            continue

        animal_means.append(
            np.nanmean(
                np.stack(valid, axis=0),
                axis=0
            )
        )

    if len(animal_means) == 0:
        return None

    return np.nanmean(
        np.stack(animal_means, axis=0),
        axis=0
    )


# ===========================================================================
# Plot helper
# ===========================================================================

def plot_ct_matrix(
    ax,
    mat,
    title='',
    vmin=0.45,
    vmax=1.0,
    cmap='RdYlGn'
):

    im = ax.imshow(
        mat,
        origin='lower',
        aspect='auto',
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        extent=[
            bin_mean[0],
            bin_mean[-1],
            bin_mean[0],
            bin_mean[-1]
        ]
    )

    ax.axhline(
        0,
        color='white',
        lw=0.8,
        linestyle='--',
        alpha=0.5
    )

    ax.axvline(
        0,
        color='white',
        lw=0.8,
        linestyle='--',
        alpha=0.5
    )

    ax.set_xlabel('test bin (s)')
    ax.set_ylabel('train bin (s)')
    ax.set_title(title)

    return im


# ===========================================================================
# Grand mean plots
# ===========================================================================

print('\nPlotting grand means...')

for method in METHODS:

    for contrast in CONTRASTS:

        fig, axes = plt.subplots(
            2,
            2,
            figsize=(12, 10)
        )

        fig.suptitle(
            f'{METHOD_LABELS[method]} — '
            f'{CONTRAST_LABELS[contrast]}'
        )

        for row, (region_label, results, animals) in enumerate([
            ('BG', bg_results, bg_animals),
            ('CB', cb_results, cb_animals)
        ]):

            for col, align in enumerate(ALIGNMENTS):

                ax = axes[row, col]

                mat = grand_mean_matrix(
                    results,
                    animals,
                    method,
                    contrast,
                    align
                )

                if mat is None:
                    ax.set_visible(False)
                    continue

                im = plot_ct_matrix(
                    ax,
                    mat,
                    title=f'{region_label} {ALIGN_LABELS[align]}'
                )

                plt.colorbar(im, ax=ax)

        plt.tight_layout()

        fname = os.path.join(
            SAVE_DIR,
            'comparison',
            f'grand_{method}_{contrast}.png'
        )

        fig.savefig(fname, dpi=150)
        plt.close(fig)


# ===========================================================================
# Statistics
# ===========================================================================

print('\nComputing statistics...')

rows = []

for region_label, results, animals in [
    ('BG', bg_results, bg_animals),
    ('CB', cb_results, cb_animals)
]:

    for method in METHODS:

        for contrast in CONTRASTS:

            for align in ALIGNMENTS:

                diag_vals = []
                off_vals  = []

                for animal in animals:

                    sess_mats = results[
                        animal
                    ][method][contrast][align]

                    valid = [
                        m for m in sess_mats
                        if m is not None
                    ]

                    if len(valid) == 0:
                        continue

                    mat = np.nanmean(
                        np.stack(valid, axis=0),
                        axis=0
                    )

                    diag_vals.append(
                        np.nanmean(np.diag(mat))
                    )

                    mask = ~np.eye(
                        mat.shape[0],
                        dtype=bool
                    )

                    off_vals.append(
                        np.nanmean(mat[mask])
                    )

                if len(diag_vals) == 0:
                    continue

                try:

                    stat, pval = stats.wilcoxon(
                        diag_vals,
                        off_vals,
                        alternative='greater'
                    )

                except Exception:

                    stat = np.nan
                    pval = np.nan

                rows.append({

                    'region': region_label,

                    'method': METHOD_LABELS[method],

                    'contrast': CONTRAST_LABELS[contrast],

                    'alignment': ALIGN_LABELS[align],

                    'mean_diag': np.nanmean(diag_vals),

                    'mean_offdiag': np.nanmean(off_vals),

                    'diag_minus_offdiag':
                        np.nanmean(diag_vals)
                        -
                        np.nanmean(off_vals),

                    'wilcoxon_stat': stat,

                    'p_value': pval,

                    'significant':
                        bool(pval < 0.05)
                        if not np.isnan(pval)
                        else False,

                    'n_animals': len(diag_vals)
                })

df = pd.DataFrame(rows)

csv_path = os.path.join(
    SAVE_DIR,
    'stats',
    'cross_temporal_stats.csv'
)

df.to_csv(
    csv_path,
    index=False,
    float_format='%.4f'
)

print(f'Saved stats → {csv_path}')

print('\nDiagonal > Off-diagonal')

if len(df) == 0:

    print('No valid results.')

else:

    sig = df[df['significant']]

    if len(sig) == 0:

        print('No significant dynamic coding.')

    else:

        for _, r in sig.iterrows():

            print(
                f"{r['region']:3s} "
                f"{r['contrast']:12s} "
                f"{r['alignment']:6s} "
                f"{r['method']:12s} "
                f"diag={r['mean_diag']:.3f} "
                f"off={r['mean_offdiag']:.3f} "
                f"p={r['p_value']:.4f}"
            )

print('\nDONE')